# Load Fine-Tuned Virtual Twin Model

This notebook loads the already-downloaded MLX base model with the saved LoRA adapter. It does not download or train anything.

In [ ]:
from pathlib import Path
import json
from datetime import datetime

from mlx_lm import generate, load
from mlx_lm.sample_utils import make_sampler

PROJECT_ROOT = Path.cwd().parent
BASE_MODEL_DIR = PROJECT_ROOT / "models" / "base" / "mlx-community__Qwen3-4B-4bit"
ADAPTER_DIR = PROJECT_ROOT / "adapters" / "virtual_twin_style"
ADAPTER_FILE = ADAPTER_DIR / "adapters.safetensors"
ADAPTER_CONFIG = ADAPTER_DIR / "adapter_config.json"

if not BASE_MODEL_DIR.exists():
    raise FileNotFoundError(f"Base model not found: {BASE_MODEL_DIR}")

if not ADAPTER_FILE.exists():
    raise FileNotFoundError(f"LoRA adapter file not found: {ADAPTER_FILE}")

print("Base model:", BASE_MODEL_DIR)
print("LoRA adapter:", ADAPTER_FILE)
print("Adapter updated:", datetime.fromtimestamp(ADAPTER_FILE.stat().st_mtime))


## Load Model + LoRA Adapter

This loads the local base model and applies the saved LoRA adapter. Use this when you want to keep the adapter separate from the base model.

In [ ]:
model, tokenizer = load(str(BASE_MODEL_DIR), adapter_path=str(ADAPTER_DIR))
sampler = make_sampler(temp=0.5)

print("Fine-tuned model loaded with LoRA adapter.")

## Quick Test

In [ ]:
SYSTEM_MESSAGE = (
    "You are Mohamad's professional virtual assistant. You do not pretend to be Mohamad. "
    "Answer clearly, concisely, and only make claims supported by available context. "
    "When the user's request is vague or missing important context, ask one concise follow-up question before answering. Do not guess."
)

messages = [
    {"role": "system", "content": SYSTEM_MESSAGE},
    {"role": "user", "content": "Can Mohamad help me?"},
]

prompt = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True,
)

response = generate(
    model,
    tokenizer,
    prompt=prompt,
    sampler=sampler,
    max_tokens=96,
)

print(response)

## Adapter Check

Use this to confirm which LoRA adapter was trained and loaded.

## Compare Base vs LoRA

If these answers are almost identical, the adapter is probably too weak rather than missing. The current adapter was trained with a tiny dataset and only 100 iterations.

In [ ]:
base_model, base_tokenizer = load(str(BASE_MODEL_DIR))

comparison_messages = [
    {"role": "system", "content": SYSTEM_MESSAGE},
    {"role": "user", "content": "What is the capital of Japan?"},
]

base_prompt = base_tokenizer.apply_chat_template(
    comparison_messages,
    tokenize=False,
    add_generation_prompt=True,
)

adapter_prompt = tokenizer.apply_chat_template(
    comparison_messages,
    tokenize=False,
    add_generation_prompt=True,
)

print("BASE MODEL:")
print(generate(base_model, base_tokenizer, prompt=base_prompt, sampler=sampler, max_tokens=96))

print("\nLORA ADAPTER:")
print(generate(model, tokenizer, prompt=adapter_prompt, sampler=sampler, max_tokens=96))


## Chat Test

Type `exit` or `quit` to stop.

In [ ]:
chat_messages = [{"role": "system", "content": SYSTEM_MESSAGE}]

while True:
    user_input = input("Write your query: ").strip()

    if user_input.lower() in {"exit", "quit"}:
        break

    if not user_input:
        continue

    chat_messages.append({"role": "user", "content": user_input})

    chat_prompt = tokenizer.apply_chat_template(
        chat_messages,
        tokenize=False,
        add_generation_prompt=True,
    )

    assistant_response = generate(
        model,
        tokenizer,
        prompt=chat_prompt,
        sampler=sampler,
        max_tokens=128,
    )

    print(assistant_response)
    chat_messages.append({"role": "assistant", "content": assistant_response})